### Notebook to tweek YAIB's preprocessing of the data, to fit a SSL setup
- This means not having labels


#### YAIB preprocessing pipeline

raw data -> split data (funciton) -> preprocess data (class)

In [23]:
import os
import copy
import logging

import gin
import json
import hashlib
import pandas as pd
import polars as pl
from pathlib import Path
import pickle
from timeit import default_timer as timer
from sklearn.model_selection import StratifiedKFold, KFold, StratifiedShuffleSplit, ShuffleSplit
from icu_benchmarks.data.preprocessor import Preprocessor, PandasClassificationPreprocessor, PolarsClassificationPreprocessor
from icu_benchmarks.constants import RunMode
from icu_benchmarks.run_utils import check_required_keys
from icu_benchmarks.data.constants import DataSplit as Split, DataSegment as Segment, VarType as Var


In [24]:
from icu_benchmarks.data.split_process_data import *
from icu_benchmarks.cross_validation import execute_repeated_cv  # adjust if path is different
from icu_benchmarks.run import *
import gin

# Path ti configs
#'os.chdir("/work3/s185395/YAIB/")

# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/Regression.gin")


ParsedConfigFileIncludesAndImports(filename='/work3/s185395/YAIB/configs/tasks/Regression.gin', imports=[], includes=[ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Imports.gin', imports=['icu_benchmarks.data.split_process_data', 'icu_benchmarks.data.loader', 'icu_benchmarks.models.wrappers', 'icu_benchmarks.models.dl_models', 'icu_benchmarks.models.ml_models'], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/PredictionTaskVariables.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/CrossValidation.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Dataloader.gin', imports=[], includes=[])])

In [56]:
# Override the outcome scaling range
gin.bind_parameter("base_regression_preprocessor.outcome_min", 0)
gin.bind_parameter("base_regression_preprocessor.outcome_max", 10)

# Call the preprocessing function with the new scale
data = preprocess_data(
    data_dir=Path("demo_data/los/mimic_demo"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = False,
    runmode=RunMode.regression,
)

/work3/s185395/YAIB/yaib_venv_imputation/lib/python3.10/site-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/work3/s185395/YAIB/yaib_venv_imputation/lib/python3.10/site-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/work3/s185395/YAIB/yaib_venv_imputation/lib/python3.10/site-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [57]:
print(type(data['train']['FEATURES']))
print(data['train']['FEATURES'].columns)

<class 'polars.dataframe.frame.DataFrame'>
['stay_id', 'time', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'MissingIndicator_alb', 'MissingIndicator_alp', 'MissingIndicator_alt', 'MissingIndicator_ast', 'MissingIndicator_be', 'MissingIndicator_bicar', 'MissingIndicator_bili', 'MissingIndicator_bili_dir', 'MissingIndicator_bnd', 'MissingIndicator_bun', 'MissingIndicator_ca', 'MissingIndicator_cai', 'MissingIndicator_ck', 'MissingIndicator_ckmb', 'MissingIndicator_cl', 'MissingIndicator_crea', 'MissingIndicator_crp', 'MissingIndicator_dbp', 'MissingIndicator_fgn', 'MissingIndicator_fio2', 'MissingIndicator_glu', 'MissingIndicator_hgb', 'MissingIndicator_hr', 'MissingIndicator_inr_pt

In [54]:
data['train']['OUTCOME'].shape

(6316, 3)

#### Next step will try to be to build a dataset class that 
1. takes input like the polardataset classes in YAIB
2. computes the output like the MortalityDataset does it for R's repo 
Then we make sure that the raw datastructures can be preprocessed by YAIB's existing tools and that when the preprocessed data is loaded that it can be inputted into R's implementation 

- Maybe be aware of the paired aspect R did in her work. How is class impalance being handled by YAIB? 

This is what Chat says for now, check conversation again when working 

Polars Input
  → group by stay_id
  → sort by time
  → create (T, F) arrays
  → compute masks
  → compute deltas
  → output (data, times, static, label, mask, delta) per patient


## Understanding PredictionPolarsDataset input and output structure

In [ ]:
## Input, after preprocessing 

# Data dict main structure 
data.keys()
dict_keys(['train', 'val', 'test'])

# Within each split 
data['train'].keys()
dict_keys(['OUTCOME', 'FEATURES'])

# Outcome structure
print(type(data['train']['OUTCOME']))
print(data['train']['OUTCOME'].columns)
<class 'polars.dataframe.frame.DataFrame'>
['stay_id', 'time', 'label']
[num_samples x 3]

# Feature structure 
print(type(data['train']['FEATURES']))
print(data['train']['FEATURES'].columns)
<class 'polars.dataframe.frame.DataFrame'>
['stay_id', 'time', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'MissingIndicator_alb', 'MissingIndicator_alp', 'MissingIndicator_alt', 'MissingIndicator_ast', 'MissingIndicator_be', 'MissingIndicator_bicar', 'MissingIndicator_bili', 'MissingIndicator_bili_dir', 'MissingIndicator_bnd', 'MissingIndicator_bun', 'MissingIndicator_ca', 'MissingIndicator_cai', 'MissingIndicator_ck', 'MissingIndicator_ckmb', 'MissingIndicator_cl', 'MissingIndicator_crea', 'MissingIndicator_crp', 'MissingIndicator_dbp', 'MissingIndicator_fgn', 'MissingIndicator_fio2', 'MissingIndicator_glu', 'MissingIndicator_hgb', 'MissingIndicator_hr', 'MissingIndicator_inr_pt', 'MissingIndicator_k', 'MissingIndicator_lact', 'MissingIndicator_lymph', 'MissingIndicator_map', 'MissingIndicator_mch', 'MissingIndicator_mchc', 'MissingIndicator_mcv', 'MissingIndicator_methb', 'MissingIndicator_mg', 'MissingIndicator_na', 'MissingIndicator_neut', 'MissingIndicator_o2sat', 'MissingIndicator_pco2', 'MissingIndicator_ph', 'MissingIndicator_phos', 'MissingIndicator_plt', 'MissingIndicator_po2', 'MissingIndicator_ptt', 'MissingIndicator_resp', 'MissingIndicator_sbp', 'MissingIndicator_temp', 'MissingIndicator_tnt', 'MissingIndicator_urine', 'MissingIndicator_wbc']
[num_samples x 98]

stay_id,time,alb,alp,alt,ast,be,bicar,bili,bili_dir,bnd,bun,ca,cai,ck,ckmb,cl,crea,crp,dbp,fgn,fio2,glu,hgb,hr,inr_pt,k,lact,lymph,map,mch,mchc,mcv,methb,mg,na,neut,…,MissingIndicator_cai,MissingIndicator_ck,MissingIndicator_ckmb,MissingIndicator_cl,MissingIndicator_crea,MissingIndicator_crp,MissingIndicator_dbp,MissingIndicator_fgn,MissingIndicator_fio2,MissingIndicator_glu,MissingIndicator_hgb,MissingIndicator_hr,MissingIndicator_inr_pt,MissingIndicator_k,MissingIndicator_lact,MissingIndicator_lymph,MissingIndicator_map,MissingIndicator_mch,MissingIndicator_mchc,MissingIndicator_mcv,MissingIndicator_methb,MissingIndicator_mg,MissingIndicator_na,MissingIndicator_neut,MissingIndicator_o2sat,MissingIndicator_pco2,MissingIndicator_ph,MissingIndicator_phos,MissingIndicator_plt,MissingIndicator_po2,MissingIndicator_ptt,MissingIndicator_resp,MissingIndicator_sbp,MissingIndicator_temp,MissingIndicator_tnt,MissingIndicator_urine,MissingIndicator_wbc
i64,duration[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
201006,1d 10h,-0.60284,0.0,0.0,0.0,-1.290317,-1.086113,0.0,0.0,-0.356632,-0.266678,-1.49822,-0.103355,0.0,0.0,0.280377,0.062211,0.0,-0.172791,0.0,2.424732,-0.793612,-1.201038,0.573495,-0.580126,-0.211663,-0.703456,-0.61205,-0.418339,-0.556371,-1.225902,0.201866,0.0,0.543222,-0.668656,-0.08987,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,false,true
201006,3d 18h,-0.60284,0.0,0.0,0.0,0.206217,0.408526,0.0,0.0,-0.671308,0.369555,-0.99231,0.819151,0.0,0.0,-0.69228,0.667675,0.0,1.135932,0.0,0.647109,0.875391,-0.727286,-0.617305,-0.580126,-0.211663,-0.593434,-0.61205,1.328718,-0.556371,-0.748543,-0.101033,0.0,0.327658,-1.040809,0.471718,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,false,true,false,true
201006,5d 9h,-0.60284,0.0,0.0,0.0,-0.292628,-0.151964,0.0,0.0,-0.671308,1.080638,0.145987,0.726901,0.0,0.0,-0.275427,0.667675,0.0,-0.03503,0.0,0.393163,-0.218577,0.69397,0.898258,-0.580126,-0.658271,-0.730962,-0.427162,0.034602,-0.266838,0.027165,-0.252482,0.0,0.543222,-1.226885,0.312348,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,false,true,false,true


In [71]:
data['train']['OUTCOME'].filter(pl.col('stay_id') == 201006)

stay_id,time,label
i64,duration[ms],f64
201006,0ms,1.68
201006,1h,1.68
201006,2h,1.68
201006,3h,1.67
201006,4h,1.66
…,…,…
201006,6d 20h,0.06
201006,6d 21h,0.05
201006,6d 22h,0.04


In [73]:
data['train']['FEATURES'].filter(pl.col('stay_id') == 201006).sort('time')

stay_id,time,alb,alp,alt,ast,be,bicar,bili,bili_dir,bnd,bun,ca,cai,ck,ckmb,cl,crea,crp,dbp,fgn,fio2,glu,hgb,hr,inr_pt,k,lact,lymph,map,mch,mchc,mcv,methb,mg,na,neut,…,MissingIndicator_cai,MissingIndicator_ck,MissingIndicator_ckmb,MissingIndicator_cl,MissingIndicator_crea,MissingIndicator_crp,MissingIndicator_dbp,MissingIndicator_fgn,MissingIndicator_fio2,MissingIndicator_glu,MissingIndicator_hgb,MissingIndicator_hr,MissingIndicator_inr_pt,MissingIndicator_k,MissingIndicator_lact,MissingIndicator_lymph,MissingIndicator_map,MissingIndicator_mch,MissingIndicator_mchc,MissingIndicator_mcv,MissingIndicator_methb,MissingIndicator_mg,MissingIndicator_na,MissingIndicator_neut,MissingIndicator_o2sat,MissingIndicator_pco2,MissingIndicator_ph,MissingIndicator_phos,MissingIndicator_plt,MissingIndicator_po2,MissingIndicator_ptt,MissingIndicator_resp,MissingIndicator_sbp,MissingIndicator_temp,MissingIndicator_tnt,MissingIndicator_urine,MissingIndicator_wbc
i64,duration[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
201006,0ms,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.516011,0.0,0.0,0.454634,-0.930323,0.681749,-0.580126,-0.658271,0.0,-0.61205,0.422837,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,false,false,true,false,true,true,false,false,false,false,false,true,false,false,false,false,false,true,false,false,false,false,true,true,false,false,true,false,false,false,true,true,true,false
201006,1h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.619331,0.0,0.0,0.454634,-0.930323,0.925322,-0.580126,-0.658271,0.0,-0.61205,0.735582,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,false,true
201006,2h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.791532,0.0,0.0,0.454634,-0.930323,0.925322,-0.580126,-0.658271,0.0,-0.61205,0.90813,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,false,true
201006,3h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.860412,0.0,0.0,0.454634,-0.930323,0.844131,-0.580126,-0.658271,0.0,-0.61205,1.199306,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,false,true,true,true
201006,4h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.17161,0.0,0.0,0.454634,-0.930323,0.140477,-0.580126,-0.658271,0.0,-0.61205,0.034602,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
201006,6d 20h,-0.60284,0.0,0.0,0.0,-1.290317,0.221696,0.0,0.0,-0.041957,1.679445,1.03133,2.202911,0.0,0.0,-0.69228,1.108013,0.0,1.067052,0.0,2.424732,-0.639334,0.761649,1.331277,-0.580126,2.467983,-0.758467,-1.038713,0.68166,

## Mortality dataset data output

In [ ]:
# I need to create all of these 6 from the preprocessed data format above 

Dynamic Features 
[batch_size, num_sensors, num_timesteps]

Static Features
[batch_size, num_static_features]

label_array
[batch_size, 1]

Sensor Mask
[batch_size, num_sensors, num_timesteps]

Time Features
[batch_size, num_timesteps]

Delta Features
[batch_size, num_sensors, num_timesteps]

In [ ]:
# First version of dataset class
# does not handle missing values propperly by using missingness columns 
# Be aware of padding in batch, and also missingness indicators. Are they distinguished between? 
    # Check how R does it in her dataset class, to me it looks like she just combines them so that 
    # ... padding is just the same type of missingness as missing values? 


import torch
import numpy as np
import polars as pl
from torch import Tensor
from typing import Tuple
from torch.utils.data import Dataset


class CustomImputationDataset(CommonPolarsDataset):
    def __init__(self, data: Dict[str, pl.DataFrame], split: str = "train", max_length=2881):
        """
        Initialize the dataset class and load the data.

        Arguments:
            data: A dictionary of Polars DataFrames with keys "dynamic", "static", and "outcome".
            split: The data split (train/val/test).
            max_length: Maximum sequence length for padding.
        """
        # Initialize the base class (CommonPolarsDataset)
        super().__init__(data, split=split, vars=vars)

        # Store max_length for padding
        self.max_length = max_length

    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor, Tensor, Tensor]:
        """
        Function to sample from the data split of choice. Used for deep learning implementations.

        Args:
            idx: A specific row index to sample.

        Returns:
            A sample from the data, consisting of data, labels, padding mask, and other arrays.
        """
        # Extracting the stay_id for the specific index
        stay_id = self.outcome_df[self.vars["GROUP"]].unique()[idx]

        # Filter the data to get the relevant data for this stay_id
        window = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(pl.exclude(self.vars["GROUP"])).to_numpy()
        labels = self.outcome_df.filter(pl.col(self.vars["GROUP"]) == stay_id)[self.vars["LABEL"]].to_numpy().astype(float)

        # Handling the case where only one label exists
        if len(labels) == 1:
            # only one label per stay, align with window
            labels = np.concatenate([np.empty(window.shape[0] - 1) * np.nan, labels], axis=0)

        # Padding the data to match the max length
        length_diff = self.max_length - window.shape[0]
        pad_mask = np.ones(window.shape[0])  # Initially, set all to 1

        if length_diff > 0:
            # If the window is shorter than max_length, pad it with zeros
            window = np.concatenate([window, np.ones((length_diff, window.shape[1])) * 0.0], axis=0)
            labels = np.concatenate([labels, np.ones(length_diff) * 0.0], axis=0)
            pad_mask = np.concatenate([pad_mask, np.zeros(length_diff)], axis=0)

        # Handling missing labels
        not_labeled = np.argwhere(np.isnan(labels))
        if len(not_labeled) > 0:
            labels[not_labeled] = -1  # Mark missing labels as -1
            pad_mask[not_labeled] = 0  # Corresponding mask as 0

        # Convert all to the appropriate types
        pad_mask = pad_mask.astype(bool)
        labels = labels.astype(np.float32)
        data = window.astype(np.float32)

        # Additional arrays needed for the restructured data
        times = window[:, 0]  # Assuming the first column is the time
        static = np.zeros_like(window)  # Assuming static features are zero, you should adapt this based on your dataset
        delta = np.diff(times, prepend=0)  # Time difference between readings

        # Return all of the required arrays as tensors
        return (
            torch.from_numpy(data),  # Data tensor
            torch.from_numpy(labels),  # Label tensor
            torch.from_numpy(pad_mask),  # Pad mask tensor
            torch.from_numpy(times),  # Time tensor (needs adjustment based on actual data)
            torch.from_numpy(static),  # Static tensor (needs actual static features)
            torch.from_numpy(delta)   # Delta tensor (time difference)
        )

    def __len__(self) -> int:
        """
        Return the total number of samples in the dataset.
        """
        return len(self.outcome_df[self.vars["GROUP"]])

    def to_tensor(self):
        """
        Convert the data to tensors. This could be used to return the entire dataset in tensor format.
        """
        return [
            self.data_array,
            self.sensor_mask_array,
            self.times_array,
            self.static_array,
            self.label_array,
            self.delta_array,
        ]
